<a href="https://colab.research.google.com/github/RealGoldenGeneral/SmartVerify-Take-Home-Assignment/blob/main/notebooks/Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extended Dataset Analysis
Before creating the pipeline, we would need to understand both schemas. In this dataset analysis we are looking for class imbalance, outliers, and data types. This analysis will be important when analyzing features to input to the model.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
# Load datasets
df_query = pd.read_csv("../data/query_events.csv")
df_sessions = pd.read_csv("../data/sessions.csv")

print("Size of query events dataset:", len(df_query))
print("Size of sessions dataset:", len(df_sessions))

print("Columns of query events dataset:", df_query.columns)
print("Columns of sessions dataset:", df_sessions.columns)

Size of query events dataset: 3309
Size of sessions dataset: 230
Columns of query events dataset: Index(['session_id', 'timestamp', 'enterprise_id', 'user_id', 'agent_id',
       'query_text', 'query_type', 'table_or_tables_accessed', 'row_estimate',
       'model_confidence', 'intent', 'status_code'],
      dtype='object')
Columns of sessions dataset: Index(['session_id', 'enterprise_id', 'user_id', 'agent_id', 'agent_type',
       'user_role', 'start_time', 'end_time', 'duration_seconds', 'num_events',
       'declared_intent', 'is_anomalous_session', 'slow_drip_exfiltration',
       'broad_retrieval_or_table_scan', 'intent_drift',
       'privilege_escalation_or_sensitive_access_violation',
       'high_entropy_exploration',
       'off_hours_sensitive_or_high_volume_access'],
      dtype='object')


,session_id,enterprise_id,user_id,agent_id,agent_type,user_role,start_time,end_time,duration_seconds,num_events,declared_intent,is_anomalous_session,slow_drip_exfiltration,broad_retrieval_or_table_scan,intent_drift,privilege_escalation_or_sensitive_access_violation,high_entropy_exploration,off_hours_sensitive_or_high_volume_access
0,sess_1000,ent_nexus_beta,usr_500,agt_6,automated_cron,analyst,2026-05-17 07:47:53,2026-05-17 08:25:43,2270,13,inventory_check,0,0,0,0,0,0,0
1,sess_1001,ent_corp_alpha,usr_501,agt_2,internal_tool,admin,2026-05-07 19:54:19,2026-05-07 20:22:05,1666,15,system_health_check,0,0,0,0,0,0,0
2,sess_1002,ent_nexus_beta,usr_502,agt_6,user_facing_llm,analyst,2026-05-12 17:35:46,2026-05-12 17:59:21,1415,17,billing_summary,0,0,0,0,0,0,0
3,sess_1003,ent_corp_alpha,usr_503,agt_6,user_facing_llm,support,2026-05-20 15:15:01,2026-05-20 16:01:57,2816,20,order_management,0,0,0,0,0,0,0
4,sess_1004,ent_corp_alpha,usr_504,agt_11,internal_tool,admin,2026-05-19 12:20:54,2026-05-19 12:53:01,1927,14,report_generation,0,0,0,0,0,0,0


In [8]:
df_query.head()

,session_id,timestamp,enterprise_id,user_id,agent_id,query_text,query_type,table_or_tables_accessed,row_estimate,model_confidence,intent,status_code
0,sess_1000,2026-05-17 07:49:19,ent_nexus_beta,usr_500,agt_6,"SELECT warehouse, updated_at FROM inventory WH...",SELECT,inventory,2320,0.8844,inventory_check,200
1,sess_1000,2026-05-17 08:03:35,ent_nexus_beta,usr_500,agt_6,"SELECT sku, AVG(amount) FROM inventory GROUP B...",AGGREGATION,inventory,5493,0.8811,inventory_check,200
2,sess_1000,2026-05-17 08:05:12,ent_nexus_beta,usr_500,agt_6,"SELECT sku, MAX(id) FROM inventory GROUP BY sku",AGGREGATION,inventory,8967,0.6200,inventory_check,200
3,sess_1000,2026-05-17 08:05:21,ent_nexus_beta,usr_500,agt_6,"SELECT id, category FROM products WHERE id = 4777",SELECT,products,241,0.8277,inventory_check,200
4,sess_1000,2026-05-17 08:14:52,ent_nexus_beta,usr_500,agt_6,"SELECT products.id, inventory.id FROM products...",JOIN,"products,inventory",88,0.7737,inventory_check,200


In [9]:
df_sessions.head()

,session_id,enterprise_id,user_id,agent_id,agent_type,user_role,start_time,end_time,duration_seconds,num_events,declared_intent,is_anomalous_session,slow_drip_exfiltration,broad_retrieval_or_table_scan,intent_drift,privilege_escalation_or_sensitive_access_violation,high_entropy_exploration,off_hours_sensitive_or_high_volume_access
0,sess_1000,ent_nexus_beta,usr_500,agt_6,automated_cron,analyst,2026-05-17 07:47:53,2026-05-17 08:25:43,2270,13,inventory_check,0,0,0,0,0,0,0
1,sess_1001,ent_corp_alpha,usr_501,agt_2,internal_tool,admin,2026-05-07 19:54:19,2026-05-07 20:22:05,1666,15,system_health_check,0,0,0,0,0,0,0
2,sess_1002,ent_nexus_beta,usr_502,agt_6,user_facing_llm,analyst,2026-05-12 17:35:46,2026-05-12 17:59:21,1415,17,billing_summary,0,0,0,0,0,0,0
3,sess_1003,ent_corp_alpha,usr_503,agt_6,user_facing_llm,support,2026-05-20 15:15:01,2026-05-20 16:01:57,2816,20,order_management,0,0,0,0,0,0,0
4,sess_1004,ent_corp_alpha,usr_504,agt_11,internal_tool,admin,2026-05-19 12:20:54,2026-05-19 12:53:01,1927,14,report_generation,0,0,0,0,0,0,0


In [15]:
df_query.info()

df_sessions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3309 entries, 0 to 3308
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   session_id                3309 non-null   object 
 1   timestamp                 3309 non-null   object 
 2   enterprise_id             3309 non-null   object 
 3   user_id                   3309 non-null   object 
 4   agent_id                  3309 non-null   object 
 5   query_text                3309 non-null   object 
 6   query_type                3309 non-null   object 
 7   table_or_tables_accessed  3309 non-null   object 
 8   row_estimate              3309 non-null   int64  
 9   model_confidence          3309 non-null   float64
 10  intent                    3309 non-null   object 
 11  status_code               3309 non-null   int64  
dtypes: float64(1), int64(2), object(9)
memory usage: 310.3+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 230 entri

In [13]:
df_query.describe()

,row_estimate,model_confidence,status_code
count,3309.000000,3309.000000,3309.000000
mean,2948.741009,0.837657,209.696585
std,13033.965041,0.133519,49.464586
min,10.000000,0.300000,200.000000
25%,172.000000,0.762000,200.000000
50%,578.000000,0.863200,200.000000
75%,2093.000000,0.941600,200.000000
max,373919.000000,0.995000,500.000000


In [14]:
df_sessions.describe()

,duration_seconds,num_events,is_anomalous_session,slow_drip_exfiltration,broad_retrieval_or_table_scan,intent_drift,privilege_escalation_or_sensitive_access_violation,high_entropy_exploration,off_hours_sensitive_or_high_volume_access
count,230.000000,230.000000,230.000000,230.000000,230.000000,230.000000,230.000000,230.000000,230.000000
mean,1768.508696,14.386957,0.221739,0.047826,0.043478,0.056522,0.056522,0.034783,0.034783
std,1652.860867,6.013290,0.416323,0.213864,0.204376,0.231430,0.231430,0.183628,0.183628
min,458.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1089.750000,11.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1403.500000,14.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1769.500000,16.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,10111.000000,45.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
